# NB_lab_test_trends
Builds `caresync_databricks.gold.lab_test_trends` per STTM sheet **lab_test_trends**.
* **Grain:** one row per test_name per calendar month
* **Driving table:** `silver.lab_results` grouped by test_name, year, month of test_date
* **Joins:** optional LEFT JOIN calendar on the first day of the month (for month_name, quarter_name)
* **Window:** LAG(abnormal_rate_pct) by test_name ordered by year, month (month-over-month trend)
* **Load:** FULL, CREATE OR REPLACE TABLE ... AS SELECT
* **Data note:** the actual calendar table is `silver.calender` with INT key `datekey`. The STTM calls it `silver.calendar` / `date_key`.

In [0]:
%sql
CREATE OR REPLACE TABLE caresync_databricks.gold.lab_test_trends
COMMENT 'Gold: one row per test_name per month with abnormal/critical rates and month-over-month trend (STTM lab_test_trends)'
AS
WITH monthly AS (
  SELECT
    test_name,
    year(test_date)                       AS year,
    month(test_date)                      AS month,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM caresync_databricks.silver.lab_results
  GROUP BY test_name, year(test_date), month(test_date)
),
rates AS (
  SELECT
    m.*,
    CAST(abnormal_test_count / total_tests * 100 AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(critical_test_count / total_tests * 100 AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  w.test_name,
  CAST(w.year AS INT)                                                     AS year,
  CAST(w.month AS INT)                                                    AS month,
  c.month_name,
  c.quarter_name,
  CAST(w.total_tests AS INT)                                              AS total_tests,
  CAST(w.abnormal_test_count AS INT)                                      AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                      AS critical_test_count,
  w.abnormal_rate_pct,
  w.critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                    AS prev_month_abnormal_rate_pct,
  CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN caresync_databricks.silver.calender c   -- STTM: silver.calendar.date_key on first day of month
  ON c.datekey = CAST(date_format(make_date(w.year, w.month, 1), 'yyyyMMdd') AS INT);

In [0]:
NB_lab_test_records_df = spark.table("caresync_databricks.gold.hospital_daily_summary")
record_count_test_trends = NB_lab_test_records_df.count()
dbutils.notebook.exit(str(record_count_test_trends))